# Retinal Vessel Segmentation

**Group:**
* Jakub Biernat 160248
* Eryk Masian 160228

**Technologies Used:**
* **Language:** Python
* **Libraries:** TODO

## Imports

In [11]:
#General imports
from skimage import io
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score
from imblearn.metrics import geometric_mean_score, specificity_score, sensitivity_score

#Imports for Retinal Vessel Detection via Image Processing
from skimage.filters import threshold_otsu, gaussian, sobel
from skimage.morphology import opening, closing
from skimage.color import rgb2gray
from skimage import exposure

## Images
From HRF image database: https://www5.cs.fau.de/research/data/fundus-images/

In [12]:
image_names = [f"{str(i).zfill(2)}_{suffix}" for i in range(1, 16) for suffix in ["h", "g", "dr"]]

def load_images(image_name):
    raw_image = io.imread(f"../data/images/{image_name}.jpg")
    gs_image = io.imread(f"../data/goldstandard/{image_name}.tif")
    mask = io.imread(f"../data/fovs/{image_name}_mask.tif")
    mask = mask[..., 0]
    return raw_image, gs_image, mask


## Quality metrics and visualisation

In [13]:
def generate_overlay(raw_image, mask_image):
    pred = mask_image > 0

    overlay = raw_image.copy()
    overlay[pred] = [0, 255, 0]

    return overlay

def calculate_metrics(gs_image, generated_image, mask = None):
    valid = mask > 0

    y_true = (gs_image > 0)[valid].astype(int)
    y_generated = (generated_image > 0)[valid].astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_generated, labels=[0, 1]).ravel()

    accuracy = accuracy_score(y_true, y_generated)

    sensitivity = sensitivity_score(y_true, y_generated)

    specificity = specificity_score(y_true, y_generated)

    gmean = geometric_mean_score(y_true, y_generated, average='binary')

    return {
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "Accuracy": accuracy,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "G-Mean": gmean
    }

## Retinal Vessel Detection Functions

### Retinal Vessel Detection via Image processing

In [14]:
def retinal_vessel_segmentation_image_processing(image, mask):
    ######## Pre-processing ########
    image = rgb2gray(image)

    image = gaussian(image, sigma=1)

    image = exposure.equalize_hist(image)

    ######## Core processing ########
    image = sobel(image)

    ######## Post-processing ########
    image = image > threshold_otsu(image)

    image = closing(image)
    image = opening(image)

    return image

### Retinal Vessel Detection via Traditional Machine Learning

In [ ]:
#TODO

### Retinal Vessel Detection via Deep Learning

In [ ]:
#TODO

### App

In [15]:
# ---------------- UI ----------------
image_selector = widgets.Dropdown(
    options=image_names,
    description="Obraz:"
)

segment_button = widgets.Button(
    description="Segmentacja",
    button_style="success",
    icon="play"
)

view_mode = widgets.ToggleButtons(
    options=[
        ("Maski", "masks"),
        ("Nałożenia", "overlays")
    ],
    description="Widok:"
)

preview_output = widgets.Output()
result_output = widgets.Output()
metrics_output = widgets.Output()

last_result = {}

progress_bar = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description="Postęp:",
    bar_style="",
    layout=widgets.Layout(width="250px", visibility="hidden")
)

progress_label = widgets.HTML(
    value="",
    layout=widgets.Layout(visibility="hidden")
)


# ---------------- CLEANING ----------------
def on_image_change(change):
    global last_result
    last_result = {}

    with preview_output:
        clear_output(wait=True)
    with result_output:
        clear_output(wait=True)
    with metrics_output:
        clear_output(wait=True)

    show_preview()


# ---------------- PREVIEW ----------------
def show_preview(change=None):
    with preview_output:
        clear_output(wait=True)

        image_name = image_selector.value
        raw_image, _, _ = load_images(image_name)

        plt.figure(figsize=(5, 5))
        plt.imshow(raw_image)
        plt.title(image_name)
        plt.axis("off")
        plt.show()


# ---------------- RESULT RENDERING ----------------
def render_result():
    if not last_result:
        return

    raw_image = last_result["raw_image"]
    gs_image = last_result["gs_image"]
    generated_image = last_result["generated_image"]
    gs_overlay = last_result["gs_overlay"]
    generated_overlay = last_result["generated_overlay"]

    with result_output:
        clear_output(wait=True)

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        if view_mode.value == "masks":
            axes[0].imshow(raw_image)
            axes[0].set_title("Obraz wejściowy")

            axes[1].imshow(gs_image, cmap="gray")
            axes[1].set_title("Maska ekspercka")

            axes[2].imshow(generated_image, cmap="gray")
            axes[2].set_title("Wygenerowana maska")

        elif view_mode.value == "overlays":
            axes[0].imshow(raw_image)
            axes[0].set_title("Obraz wejściowy")

            axes[1].imshow(gs_overlay)
            axes[1].set_title("Nałożona maska ekspercka")

            axes[2].imshow(generated_overlay)
            axes[2].set_title("Nałożona wygenerowana maska")

        for ax in axes:
            ax.axis("off")

        plt.tight_layout()
        plt.show()


# ---------------- SEGMENTATION ----------------
def run_segmentation(button):
    global last_result

    segment_button.disabled = True

    progress_bar.layout.visibility = "visible"
    progress_label.layout.visibility = "visible"

    progress_bar.value = 0
    progress_bar.bar_style = ""
    progress_label.value = "Przygotowywanie danych..."

    try:
        image_name = image_selector.value

        progress_bar.value = 10
        progress_label.value = "Wczytywanie obrazu..."

        raw_image, gs_image, mask = load_images(image_name)

        progress_bar.value = 30
        progress_label.value = "Trwa segmentacja naczyń..."

        generated_image = retinal_vessel_segmentation_image_processing(raw_image, mask)

        progress_bar.value = 70
        progress_label.value = "Generowanie podglądu..."

        generated_overlay = generate_overlay(raw_image, generated_image)
        gs_overlay = generate_overlay(raw_image, gs_image)

        last_result = {
            "raw_image": raw_image,
            "gs_image": gs_image,
            "generated_image": generated_image,
            "generated_overlay": generated_overlay,
            "gs_overlay": gs_overlay
        }

        render_result()

        progress_bar.value = 85
        progress_label.value = "Obliczanie metryk..."

        metrics = calculate_metrics(gs_image, generated_image, mask)

        with metrics_output:
            clear_output(wait=True)

            print("Metryki:")

            for k, v in metrics.items():
                if isinstance(v, float):
                    print(f"{k:12s}: {v:.4f}")
                else:
                    print(f"{k:12s}: {v}")

        progress_bar.value = 100
        progress_bar.bar_style = "success"
        progress_label.value = "Segmentacja zakończona."

    except Exception as e:
        progress_bar.bar_style = "danger"
        progress_label.value = f"Błąd segmentacji: {e}"

    finally:
        segment_button.disabled = False


# ---------------- VIEW MODE CHANGE ----------------
def on_view_mode_change(change):
    render_result()


# ---------------- OBSERVERS ----------------
image_selector.observe(on_image_change, names="value")
view_mode.observe(on_view_mode_change, names="value")
segment_button.on_click(run_segmentation)


# ---------------- LAYOUT ----------------
left_panel = widgets.VBox([
    image_selector,
    view_mode,
    segment_button,
    progress_bar,
    progress_label
])

top_panel = widgets.HBox([
    left_panel,
    preview_output
])

display(
    widgets.VBox([
        top_panel,
        result_output,
        metrics_output
    ])
)

# pierwszy podgląd
show_preview()